# CNN-based surface crack detection in steel plates
AI for Engineers project. Dataset: NEU-DET (real hot-rolled steel images). Runs on Colab CPU or GPU.

## 0. Get the real dataset (NEU-DET) from GitHub

In [ ]:
!pip -q install opencv-python-headless scikit-learn pandas matplotlib
!git clone -q --depth 1 https://github.com/siddhartamukherjee/NEU-DET-Steel-Surface-Defect-Detection neu
!mkdir -p allraw && cp neu/IMAGES/*.jpg neu/Validation_Images/*.jpg allraw/ && ls allraw | wc -l

In [ ]:
import os, glob, hashlib, json, shutil, sys
import numpy as np, cv2, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

RAW = "allraw"
OUT = "."
FIG = os.path.join(OUT, "figures"); os.makedirs(FIG, exist_ok=True)
CLASSES = ["crazing", "inclusion", "patches", "pitted_surface", "rolled-in_scale", "scratches"]
SIZE = 128
plt.rcParams.update({"font.size": 10, "figure.dpi": 130})

## 1. Load and clean (readability, size, duplicate check)

In [ ]:
# 1. load + clean
rows, seen = [], {}
for p in sorted(glob.glob(os.path.join(RAW, "*.jpg"))):
    name = os.path.basename(p)
    cls = name.rsplit("_", 1)[0]
    img = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
    ok = img is not None
    h = hashlib.md5(open(p, "rb").read()).hexdigest()
    dup = h in seen
    seen.setdefault(h, name)
    rows.append(dict(file=name, path=p, label=cls, readable=ok, duplicate=dup,
                     h=img.shape[0] if ok else 0, w=img.shape[1] if ok else 0,
                     mean=float(img.mean()) if ok else np.nan, std=float(img.std()) if ok else np.nan))
df = pd.DataFrame(rows)
report = dict(total=len(df), unreadable=int((~df.readable).sum()), duplicates=int(df.duplicate.sum()),
              sizes=df.groupby(["h", "w"]).size().rename("n").reset_index().to_dict("records"))
df = df[df.readable & ~df.duplicate].reset_index(drop=True)
report["after_cleaning"] = len(df)
report["per_class"] = df.label.value_counts().reindex(CLASSES).to_dict()
report["intensity"] = df.groupby("label")[["mean", "std"]].mean().round(2).reindex(CLASSES).to_dict("index")

## 2. Data visualization (EDA)

In [ ]:
# 2. EDA figures
cnt = df.label.value_counts().reindex(CLASSES)
fig, ax = plt.subplots(figsize=(7, 3.4))
bars = ax.bar(cnt.index, cnt.values, color=["#C0392B"] + ["#7F8C8D"] * 5)
ax.bar_label(bars); ax.set_ylabel("Images"); ax.set_title("Images per class (crazing = crack class)")
plt.xticks(rotation=20); plt.tight_layout(); plt.savefig(f"{FIG}/01_class_distribution.png"); plt.close()

fig, axs = plt.subplots(len(CLASSES), 5, figsize=(8, 10))
rng = np.random.default_rng(0)
for i, c in enumerate(CLASSES):
    picks = rng.choice(df[df.label == c].path.values, 5, replace=False)
    for j, p in enumerate(picks):
        axs[i, j].imshow(cv2.imread(p, 0), cmap="gray"); axs[i, j].axis("off")
    axs[i, 0].set_title(c, loc="left", fontsize=10)
plt.tight_layout(); plt.savefig(f"{FIG}/02_sample_grid.png"); plt.close()

fig, ax = plt.subplots(figsize=(7, 3.6))
for c in CLASSES:
    hist = np.zeros(256)
    for p in df[df.label == c].path.values[:100]:
        hist += cv2.calcHist([cv2.imread(p, 0)], [0], None, [256], [0, 256]).ravel()
    ax.plot(hist / hist.sum(), label=c, lw=2.2 if c == "crazing" else 1.2)
ax.set_xlabel("Pixel intensity"); ax.set_ylabel("Fraction of pixels"); ax.set_title("Pixel intensity distribution per class")
ax.legend(fontsize=8); plt.tight_layout(); plt.savefig(f"{FIG}/03_intensity_histograms.png"); plt.close()

fig, axs = plt.subplots(1, 6, figsize=(11, 2.3))
for a, c in zip(axs, CLASSES):
    m = np.mean([cv2.imread(p, 0).astype(float) for p in df[df.label == c].path], axis=0)
    a.imshow(m, cmap="gray"); a.set_title(c, fontsize=9); a.axis("off")
plt.suptitle("Mean image per class"); plt.tight_layout(); plt.savefig(f"{FIG}/04_mean_images.png"); plt.close()

## 3. Pre-processing (blur, CLAHE, resize)

In [ ]:
# 3. pre-processing
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
def preprocess(gray):
    g = cv2.GaussianBlur(gray, (3, 3), 0)
    g = clahe.apply(g)
    return cv2.resize(g, (SIZE, SIZE), interpolation=cv2.INTER_AREA)

ex = df[df.label == "crazing"].path.values[3]
raw = cv2.imread(ex, 0); blur = cv2.GaussianBlur(raw, (3, 3), 0); cl = clahe.apply(blur); fin = cv2.resize(cl, (SIZE, SIZE), interpolation=cv2.INTER_AREA)
fig, axs = plt.subplots(2, 4, figsize=(11, 5.2))
for a, im, t in zip(axs[0], [raw, blur, cl, fin], ["1. Raw (200x200)", "2. Gaussian blur 3x3", "3. CLAHE contrast", f"4. Resized {SIZE}x{SIZE}"]):
    a.imshow(im, cmap="gray", vmin=0, vmax=255); a.set_title(t, fontsize=9); a.axis("off")
for a, im in zip(axs[1], [raw, blur, cl, fin]):
    a.hist(im.ravel(), 64, color="#34495E"); a.set_yticks([])
plt.suptitle("Pre-processing steps on a crazing (crack) image, with pixel histograms"); plt.tight_layout()
plt.savefig(f"{FIG}/05_preprocessing_steps.png"); plt.close()

## 4. Stratified 70/15/15 split and save processed dataset

In [ ]:
# 4. stratified split 70/15/15
tr, tmp = train_test_split(df, test_size=0.30, stratify=df.label, random_state=42)
va, te = train_test_split(tmp, test_size=0.50, stratify=tmp.label, random_state=42)
DS = os.path.join(OUT, "NEU_steel_crack_dataset")
if os.path.exists(DS): shutil.rmtree(DS)
split_rows = []
for split, part in [("train", tr), ("val", va), ("test", te)]:
    for _, r in part.iterrows():
        d = os.path.join(DS, "processed", split, r.label); os.makedirs(d, exist_ok=True)
        cv2.imwrite(os.path.join(d, r.file.replace(".jpg", ".png")), preprocess(cv2.imread(r.path, 0)))
        d2 = os.path.join(DS, "original", split, r.label); os.makedirs(d2, exist_ok=True)
        shutil.copy(r.path, os.path.join(d2, r.file))
        split_rows.append(dict(file=r.file, label=r.label, split=split, is_crack=int(r.label == "crazing")))
sp = pd.DataFrame(split_rows); sp.to_csv(os.path.join(DS, "labels.csv"), index=False)
report["split"] = sp.groupby(["split", "label"]).size().unstack().reindex(columns=CLASSES).to_dict("index")

fig, ax = plt.subplots(figsize=(7, 3.4))
tab = sp.groupby(["label", "split"]).size().unstack().reindex(CLASSES)[["train", "val", "test"]]
tab.plot.bar(ax=ax, color=["#2C3E50", "#7F8C8D", "#C0392B"]); ax.set_ylabel("Images"); ax.set_title("Stratified 70/15/15 split")
plt.xticks(rotation=20); plt.tight_layout(); plt.savefig(f"{FIG}/06_split.png"); plt.close()

json.dump(report, open(os.path.join(OUT, "eda_report.json"), "w"), indent=1)
print(json.dumps(report, indent=1))

## 5. Load processed data and define augmentation

In [ ]:
import os, sys, json, time, glob
import numpy as np, cv2, torch, torch.nn as nn, torch.nn.functional as F
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support, confusion_matrix,
                             roc_auc_score, classification_report, ConfusionMatrixDisplay)

torch.manual_seed(42); np.random.seed(42)
DS = "NEU_steel_crack_dataset"; OUT = "."; EPOCHS = 20
FIG = os.path.join(OUT, "figures"); os.makedirs(FIG, exist_ok=True)
CLASSES = ["crazing", "inclusion", "patches", "pitted_surface", "rolled-in_scale", "scratches"]

def load(split):
    X, y = [], []
    for i, c in enumerate(CLASSES):
        for p in sorted(glob.glob(f"{DS}/processed/{split}/{c}/*.png")):
            X.append(cv2.imread(p, 0)); y.append(i)
    X = torch.tensor(np.stack(X), dtype=torch.float32).unsqueeze(1) / 255.0
    return X, torch.tensor(y)
Xtr, ytr = load("train"); Xva, yva = load("val"); Xte, yte = load("test")
MEAN, STD = Xtr.mean().item(), Xtr.std().item()
norm = lambda x: (x - MEAN) / STD
print("train/val/test", len(ytr), len(yva), len(yte), "mean/std", round(MEAN, 4), round(STD, 4))

def augment(x):
    # random flips, 90-degree rotations, brightness/contrast jitter (on real images only)
    if np.random.rand() < 0.5: x = x.flip(3)
    if np.random.rand() < 0.5: x = x.flip(2)
    k = np.random.randint(4); x = torch.rot90(x, k, (2, 3))
    b = (torch.rand(x.size(0), 1, 1, 1) - 0.5) * 0.2
    c = 1 + (torch.rand(x.size(0), 1, 1, 1) - 0.5) * 0.3
    return ((x - 0.5) * c + 0.5 + b).clamp(0, 1)

## 6. CNN model, training and evaluation functions

In [ ]:
class SteelCNN(nn.Module):
    def __init__(s, n=6):
        super().__init__()
        def blk(i, o): return nn.Sequential(nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(),
                                            nn.Conv2d(o, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(), nn.MaxPool2d(2))
        s.features = nn.Sequential(blk(1, 32), blk(32, 64), blk(64, 128), blk(128, 256))
        s.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(0.4), nn.Linear(256, n))
    def forward(s, x): return s.head(s.features(x))

def predict(model, X):
    model.eval(); out = []
    with torch.no_grad():
        for i in range(0, len(X), 128): out.append(F.softmax(model(norm(X[i:i+128])), 1))
    return torch.cat(out).numpy()

def run(name, use_aug):
    torch.manual_seed(42); np.random.seed(42)
    model = SteelCNN(); opt = torch.optim.Adam(model.parameters(), 1e-3, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, EPOCHS)
    hist = dict(train_loss=[], val_loss=[], train_acc=[], val_acc=[]); best, best_state = -1, None
    t0 = time.time()
    for ep in range(EPOCHS):
        model.train(); perm = torch.randperm(len(ytr)); tl, tc = 0, 0
        for i in range(0, len(perm), 32):
            idx = perm[i:i+32]; xb = Xtr[idx]; yb = ytr[idx]
            if use_aug: xb = augment(xb)
            out = model(norm(xb)); loss = F.cross_entropy(out, yb)
            opt.zero_grad(); loss.backward(); opt.step()
            tl += loss.item() * len(idx); tc += (out.argmax(1) == yb).sum().item()
        sched.step()
        pv = predict(model, Xva); vl = F.nll_loss(torch.log(torch.tensor(pv) + 1e-9), yva).item(); va = (pv.argmax(1) == yva.numpy()).mean()
        hist["train_loss"].append(tl / len(ytr)); hist["train_acc"].append(tc / len(ytr)); hist["val_loss"].append(vl); hist["val_acc"].append(float(va))
        if va > best: best, best_state, best_ep = va, {k: v.clone() for k, v in model.state_dict().items()}, ep + 1
        print(f"{name} ep{ep+1:02d} loss {tl/len(ytr):.3f} acc {tc/len(ytr):.3f} | val loss {vl:.3f} acc {va:.3f}", flush=True)
    model.load_state_dict(best_state); torch.save(best_state, f"{OUT}/{name}.pth")
    return model, hist, dict(best_val_acc=float(best), best_epoch=best_ep, train_time_s=round(time.time() - t0, 1),
                             params=sum(p.numel() for p in model.parameters()))

def evaluate(model, name):
    P = predict(model, Xte); yp = P.argmax(1); yt = yte.numpy()
    p, r, f, s = precision_recall_fscore_support(yt, yp, labels=range(6), zero_division=0)
    mp, mr, mf, _ = precision_recall_fscore_support(yt, yp, average="macro", zero_division=0)
    # crack (crazing) vs everything else, as a plant alarm would use it
    bt, bp = (yt == 0).astype(int), (yp == 0).astype(int)
    bpr, brc, bf, _ = precision_recall_fscore_support(bt, bp, average="binary", zero_division=0)
    m = dict(accuracy=accuracy_score(yt, yp), macro_precision=mp, macro_recall=mr, macro_f1=mf,
             roc_auc_ovr_macro=roc_auc_score(yt, P, multi_class="ovr", average="macro"),
             crack_precision=bpr, crack_recall=brc, crack_f1=bf, crack_auc=roc_auc_score(bt, P[:, 0]),
             per_class={c: dict(precision=float(p[i]), recall=float(r[i]), f1=float(f[i]), support=int(s[i])) for i, c in enumerate(CLASSES)},
             confusion=confusion_matrix(yt, yp).tolist())
    m = {k: (round(float(v), 4) if isinstance(v, (float, np.floating)) else v) for k, v in m.items()}
    print(classification_report(yt, yp, target_names=CLASSES, digits=4))
    return m, P

## 7. Train baseline and augmented models

In [ ]:
results, hists, models = {}, {}, {}
for name, aug in [("cnn_baseline_no_aug", False), ("cnn_augmented", True)]:
    model, h, info = run(name, aug)
    m, P = evaluate(model, name)
    results[name] = dict(**info, test=m); hists[name] = h; models[name] = (model, P)

json.dump(dict(results=results, history=hists, norm=dict(mean=MEAN, std=STD)), open(f"{OUT}/results.json", "w"), indent=1)

## 8. Result plots: learning curves, confusion matrix, ROC, Grad-CAM, misclassified

In [ ]:
# figures: learning curves
fig, axs = plt.subplots(1, 2, figsize=(11, 3.8))
for name, ls in [("cnn_baseline_no_aug", "--"), ("cnn_augmented", "-")]:
    h = hists[name]; e = range(1, EPOCHS + 1); lab = "baseline" if "base" in name else "augmented"
    axs[0].plot(e, h["train_loss"], ls, color="#2C3E50", label=f"{lab} train"); axs[0].plot(e, h["val_loss"], ls, color="#C0392B", label=f"{lab} val")
    axs[1].plot(e, h["train_acc"], ls, color="#2C3E50", label=f"{lab} train"); axs[1].plot(e, h["val_acc"], ls, color="#C0392B", label=f"{lab} val")
axs[0].set_title("Loss"); axs[1].set_title("Accuracy"); axs[0].set_ylim(0, 2)
for a in axs: a.set_xlabel("Epoch"); a.legend(fontsize=8); a.grid(alpha=.3)
plt.tight_layout(); plt.savefig(f"{FIG}/07_learning_curves.png", dpi=130); plt.close()

best = max(results, key=lambda k: results[k]["test"]["macro_f1"])
model, P = models[best]; yp = P.argmax(1)
fig, ax = plt.subplots(figsize=(6.4, 5.4))
ConfusionMatrixDisplay(confusion_matrix(yte.numpy(), yp), display_labels=CLASSES).plot(ax=ax, cmap="Blues", colorbar=False, xticks_rotation=35)
ax.set_title(f"Confusion matrix on test set ({best})"); plt.tight_layout(); plt.savefig(f"{FIG}/08_confusion_matrix.png", dpi=130); plt.close()

from sklearn.metrics import roc_curve
fig, ax = plt.subplots(figsize=(5.6, 4.6))
for i, c in enumerate(CLASSES):
    fpr, tpr, _ = roc_curve((yte.numpy() == i).astype(int), P[:, i]); ax.plot(fpr, tpr, lw=2.2 if i == 0 else 1.2, label=c)
ax.plot([0, 1], [0, 1], "k:", lw=.8); ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate"); ax.set_title("ROC curves (one vs rest)"); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(f"{FIG}/09_roc_curves.png", dpi=130); plt.close()

# Grad-CAM on test crazing images
def gradcam(model, x, cls):
    model.eval(); acts = {}
    hk = model.features[-1].register_forward_hook(lambda m, i, o: acts.setdefault("a", o))
    x = norm(x).requires_grad_(False); out = model(x); hk.remove()
    a = acts["a"]; g = torch.autograd.grad(out[0, cls], a)[0]
    cam = F.relu((g.mean((2, 3), keepdim=True) * a).sum(1))[0].detach().numpy()
    cam = cv2.resize(cam, (x.shape[-1], x.shape[-2])); return cam / (cam.max() + 1e-8)
idx = [i for i in range(len(yte)) if yte[i] == 0][:4] + [i for i in range(len(yte)) if yte[i] == 5][:2]
fig, axs = plt.subplots(2, len(idx), figsize=(2.1 * len(idx), 4.4))
for j, i in enumerate(idx):
    img = Xte[i, 0].numpy(); cam = gradcam(model, Xte[i:i+1], int(yp[i]))
    axs[0, j].imshow(img, cmap="gray"); axs[0, j].set_title(f"true: {CLASSES[yte[i]]}", fontsize=8)
    axs[1, j].imshow(img, cmap="gray"); axs[1, j].imshow(cam, cmap="jet", alpha=.45); axs[1, j].set_title(f"pred: {CLASSES[yp[i]]}", fontsize=8)
    axs[0, j].axis("off"); axs[1, j].axis("off")
plt.suptitle("Grad-CAM: where the CNN looks"); plt.tight_layout(); plt.savefig(f"{FIG}/10_gradcam.png", dpi=130); plt.close()

wrong = np.where(yp != yte.numpy())[0][:8]
if len(wrong):
    fig, axs = plt.subplots(1, len(wrong), figsize=(2 * len(wrong), 2.4), squeeze=False)
    for a, i in zip(axs[0], wrong):
        a.imshow(Xte[i, 0], cmap="gray"); a.axis("off"); a.set_title(f"T:{CLASSES[yte[i]]}\nP:{CLASSES[yp[i]]}", fontsize=7)
    plt.suptitle("Misclassified test images"); plt.tight_layout(); plt.savefig(f"{FIG}/11_misclassified.png", dpi=130); plt.close()
print(json.dumps({k: {kk: v[kk] for kk in ["best_val_acc", "best_epoch", "train_time_s", "params"]} | {kk: v["test"][kk] for kk in ["accuracy", "macro_f1", "crack_recall", "crack_precision", "roc_auc_ovr_macro"]} for k, v in results.items()}, indent=1))
print("best", best, "wrong", len(np.where(yp != yte.numpy())[0]))

## 9. Show figures

In [ ]:
from IPython.display import Image, display
import glob
for f in sorted(glob.glob('figures/*.png')): print(f); display(Image(f))